# 04 — Morphology Features: D-04 Korean Surface-Grammar Layer (Scaffold + Auditable Pilot)

이 노트북은 SSOT D-04 한국어 형태소 feature만 다룬다. 형태소 분석은 tokenizer pipeline 내부 단계가 아니며, o200k_base regex chunking/BPE token 경계와 일치한다고 가정하지 않는다. **Full population 실행은 이 노트북의 범위 밖이다** — Human audit/G1 close 이후 별도로 승인·실행한다.

## 0 Contract / Scope

- Analyzer: `kiwipiepy` 기본 생성자(`Kiwi()`), custom dictionary 미사용.
- Primary feature: `morpheme_density`, `particle_ratio`, `ending_ratio`, `deriv_affix_ratio`.
- Alternative feature: `morpheme_density`, `function_morpheme_ratio`(=particle+ending), `deriv_affix_ratio`.
- POS 분류: `J*`(조사), `E*`(어미), `XSN/XSV/XSA`(파생접사) 3개 그룹만 사용한다.
- 금지: POS 전체 목록(zoo) 노출, 도메인별 custom dictionary, analyzer 정규화 텍스트를 tokenizer 입력으로 재사용.
- 이 노트북은 **scaffold + auditable pilot(~1,000행 + 50-100행 sanity sample)** 까지만 실행하고, pair당 latency로 3,836,013행 full population 예상 runtime을 보고한다.

In [1]:
from __future__ import annotations

import json

from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.morphology import (
    ANALYZER_MODEL_MANIFEST_SHA256,
    ANALYZER_MODEL_VERSION,
    ANALYZER_PACKAGE_VERSION,
    MORPHOLOGY_CONFIG_SHA256,
    execute_morphology_run,
)

FORBIDDEN_COMPONENTS = {'pos_zoo', 'custom_dictionary', 'tokenizer_reuse_of_analyzer_text', 'full_population_run'}
PAIR_REGISTRY_V002 = PROJECT_ROOT / 'data/registry/PAIR_REGISTRY_v002.parquet'
REP_FEATURES_V001 = PROJECT_ROOT / 'data/registry/REP_FEATURES_v001.parquet'
PAIR_REGISTRY_V002_POSIX = PAIR_REGISTRY_V002.as_posix()
REP_FEATURES_V001_POSIX = REP_FEATURES_V001.as_posix()
PILOT_OUTPUT = PROJECT_ROOT / '.runtime/nb04-pilot/MORPH_FEATURES_PILOT_v001.parquet'
PILOT_RUNTIME = PROJECT_ROOT / '.runtime/nb04-pilot'
MANIFEST_PATH = PROJECT_ROOT / 'outputs/manifests/MORPH_FEATURES_PILOT_MANIFEST_v001.json'
{
    'analyzer_package_version': ANALYZER_PACKAGE_VERSION,
    'analyzer_model_version': ANALYZER_MODEL_VERSION,
    'analyzer_model_manifest_sha256': ANALYZER_MODEL_MANIFEST_SHA256,
    'config_sha256': MORPHOLOGY_CONFIG_SHA256,
}

{'analyzer_package_version': '0.23.2',
 'analyzer_model_version': '0.23.0',
 'analyzer_model_manifest_sha256': '3baa52f40876b78dab7e9428f2e488ca2ae3ed6b3d813df17f72e15a61fc516a',
 'config_sha256': 'c056dd7ad82dc20e2bc71e100f513a3bda89331e77975d58062dce02b03f08ae'}

## 1 Bounded pilot (N≈1,000)

candidate cohort에서 1,000행을 처리해 schema/ratio invariant를 검증하고 pair당 latency를 측정한다.

In [2]:
if PILOT_OUTPUT.exists():
    PILOT_OUTPUT.unlink()

import datetime as dt
from zoneinfo import ZoneInfo
pilot_run_id = 'NB04_PILOT_' + dt.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%dT%H%M%S')
pilot_manifest = execute_morphology_run(
    project_root=PROJECT_ROOT,
    input_path=PAIR_REGISTRY_V002,
    output_path=PILOT_OUTPUT,
    runtime_dir=PILOT_RUNTIME,
    run_id=pilot_run_id,
    limit=1000,
)
{
    'row_count': pilot_manifest['output']['row_count'],
    'validation_status': pilot_manifest['validation_status'],
    'per_pair_latency_sec': pilot_manifest['per_pair_latency_sec'],
    'projected_full_population_3836013_rows_min': pilot_manifest['projected_full_population_3836013_rows_min'],
}

NB04:MORPHOLOGY_PILOT:   0%|          | 0/1000 [00:00<?, ?it/s]

[12:50:37 KST]
phase=NB04
stage=MORPHOLOGY_PILOT
0/1000
0.00%
throughput=NA
ETA NA
elapsed 00:00:00
RSS 2.08GiB
mem_available 8.89GiB
memory_status=OK
checkpoint=NONE


[12:50:39 KST]
phase=NB04
stage=MORPHOLOGY_PILOT
1000/1000
100.00%
throughput=736.69 items/s
ETA 00:00:00
elapsed 00:00:01
RSS 0.65GiB
mem_available 10.07GiB
memory_status=OK
checkpoint=MORPH_FEATURES_WRITTEN
[12:50:39 KST]
phase=NB04
stage=MORPHOLOGY_PILOT
1000/1000
100.00%
throughput=731.33 items/s
ETA 00:00:00
elapsed 00:00:01
RSS 0.65GiB
mem_available 10.07GiB
memory_status=OK
checkpoint=MORPH_FEATURES_WRITTEN


{'row_count': 1000,
 'validation_status': 'PASS',
 'per_pair_latency_sec': 0.0023650000000000003,
 'projected_full_population_3836013_rows_min': 151.2}

## 2 Pilot artifact validation

row count, pair_id 유일성, ratio 범위를 독립적으로 재확인한다.

In [3]:
import duckdb

con = duckdb.connect()
pilot_rel = f"read_parquet('{PILOT_OUTPUT.as_posix()}')"
checks = con.execute(f'''
    SELECT
        count(*) AS n,
        count(DISTINCT pair_id) AS distinct_pair_id,
        min(ko_particle_ratio) AS min_particle, max(ko_particle_ratio) AS max_particle,
        min(ko_ending_ratio) AS min_ending, max(ko_ending_ratio) AS max_ending
    FROM {pilot_rel}
''').fetchone()
con.close()
assert checks[0] == 1000
assert checks[1] == 1000
assert 0.0 <= checks[2] and checks[3] <= 1.0
assert 0.0 <= checks[4] and checks[5] <= 1.0
{'n': checks[0], 'distinct_pair_id': checks[1]}

{'n': 1000, 'distinct_pair_id': 1000}

## 3 Pilot manifest persistence

latency 측정치와 예상 full-population runtime을 manifest로 저장한다. **Full run은 이 시점에 실행하지 않는다.**

In [4]:
MANIFEST_PATH.write_text(json.dumps(pilot_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
{
    'manifest_path': str(MANIFEST_PATH.relative_to(PROJECT_ROOT)),
    'status': pilot_manifest['status'],
}

{'manifest_path': 'outputs/manifests/MORPH_FEATURES_PILOT_MANIFEST_v001.json',
 'status': 'PILOT_ONLY_FULL_RUN_DEFERRED'}

## 4 Quantitative pilot diagnostics

N=1,000 pilot 결과에서 Kiwi exception/zero-morpheme rate, 5개 핵심 feature의 분포(min/p01/median/p99/max), [0,1] 범위를 벗어나거나 non-finite인 행 수, analyzer throughput, RSS를 측정한다. (재계산 아님: 이미 실행된 pilot의 원문에 대해 독립적으로 재분석/aggregate만 수행)

In [5]:
con = duckdb.connect()
pilot_rows = con.execute(f'''
    SELECT m.pair_id, p.ko_text_analysis, r.ko_codepoint_count,
           m.ko_morpheme_count, m.ko_morpheme_density, m.ko_particle_ratio,
           m.ko_ending_ratio, m.ko_deriv_affix_ratio
    FROM read_parquet('{PILOT_OUTPUT.as_posix()}') m
    JOIN read_parquet('{PAIR_REGISTRY_V002_POSIX}') p USING (pair_id)
    JOIN read_parquet('{REP_FEATURES_V001_POSIX}') r USING (pair_id)
''').fetchall()
con.close()
{'pilot_rows_loaded': len(pilot_rows)}

{'pilot_rows_loaded': 1000}

In [6]:
import time
import psutil
import numpy as np
from tokenization_premium.morphology import get_kiwi, morphology_features, MorphologyInputError

kiwi = get_kiwi()
exception_count = 0
zero_morpheme_count = 0
process = psutil.Process()
rss_before_gib = process.memory_info().rss / 1024**3
start = time.monotonic()
for pair_id, ko_text, cp_count, *_ in pilot_rows:
    try:
        f = morphology_features(ko_text, codepoint_count=cp_count, kiwi=kiwi)
        if f['morpheme_count'] == 0:
            zero_morpheme_count += 1
    except MorphologyInputError:
        exception_count += 1
    except Exception:
        exception_count += 1
elapsed = time.monotonic() - start
rss_after_gib = process.memory_info().rss / 1024**3
n = len(pilot_rows)
diagnostic_summary = {
    'n': n,
    'kiwi_exception_n': exception_count,
    'kiwi_exception_rate': exception_count / n,
    'zero_morpheme_n': zero_morpheme_count,
    'zero_morpheme_rate': zero_morpheme_count / n,
    'reanalysis_elapsed_sec': round(elapsed, 3),
    'reanalysis_throughput_pairs_per_sec': round(n / elapsed, 2),
    'rss_before_gib': round(rss_before_gib, 3),
    'rss_after_gib': round(rss_after_gib, 3),
}
diagnostic_summary

{'n': 1000,
 'kiwi_exception_n': 0,
 'kiwi_exception_rate': 0.0,
 'zero_morpheme_n': 0,
 'zero_morpheme_rate': 0.0,
 'reanalysis_elapsed_sec': 0.311,
 'reanalysis_throughput_pairs_per_sec': 3210.62,
 'rss_before_gib': 0.646,
 'rss_after_gib': 0.646}

In [7]:
def quantiles(values):
    arr = np.asarray(values, dtype=float)
    return {
        'min': float(np.min(arr)), 'p01': float(np.percentile(arr, 1)),
        'median': float(np.median(arr)), 'p99': float(np.percentile(arr, 99)),
        'max': float(np.max(arr)),
    }

morpheme_count_vals = [row[3] for row in pilot_rows]
morpheme_density_vals = [row[4] for row in pilot_rows]
particle_ratio_vals = [row[5] for row in pilot_rows]
ending_ratio_vals = [row[6] for row in pilot_rows]
deriv_affix_ratio_vals = [row[7] for row in pilot_rows]

distribution_report = {
    'morpheme_count': quantiles(morpheme_count_vals),
    'morpheme_density': quantiles(morpheme_density_vals),
    'particle_ratio': quantiles(particle_ratio_vals),
    'ending_ratio': quantiles(ending_ratio_vals),
    'deriv_affix_ratio': quantiles(deriv_affix_ratio_vals),
}
out_of_range = sum(
    1 for v in particle_ratio_vals + ending_ratio_vals + deriv_affix_ratio_vals if not (0.0 <= v <= 1.0)
)
non_finite = sum(
    1 for v in morpheme_density_vals + particle_ratio_vals + ending_ratio_vals + deriv_affix_ratio_vals
    if not np.isfinite(v)
)
distribution_report['ratio_outside_0_1_n'] = out_of_range
distribution_report['non_finite_n'] = non_finite
distribution_report

{'morpheme_count': {'min': 2.0,
  'p01': 3.0,
  'median': 19.0,
  'p99': 62.00999999999999,
  'max': 74.0},
 'morpheme_density': {'min': 0.35,
  'p01': 0.4,
  'median': 0.5333333333333333,
  'p99': 0.8888888888888888,
  'max': 1.6},
 'particle_ratio': {'min': 0.0,
  'p01': 0.0,
  'median': 0.15384615384615385,
  'p99': 0.300076923076923,
  'max': 0.375},
 'ending_ratio': {'min': 0.0,
  'p01': 0.0,
  'median': 0.17349498327759197,
  'p99': 0.4,
  'max': 0.5714285714285714},
 'deriv_affix_ratio': {'min': 0.0,
  'p01': 0.0,
  'median': 0.0625,
  'p99': 0.18752976190476187,
  'max': 0.25},
 'ratio_outside_0_1_n': 0,
 'non_finite_n': 0}

## 5 Morphology sanity audit sample (50-100 rows)

목적: gold morphology corpus 생성이 아니라 Kiwi analyzer 동작과 POS 매핑 구현의 sanity audit이다. domain(general/dialogue/technology/other), 길이 극단(Q1/Q5), 영어 혼용, 숫자, 그리고 pilot에서 관측된 feature extreme(밀도/조사/어미/파생접사 비율 상위)을 섞어 구성한다. 새 복잡한 quota system은 만들지 않고 단순 SQL 조건 + 상위 N만 사용한다.

In [8]:
con = duckdb.connect()
pilot_joined = (
    "(SELECT m.pair_id, p.ko_text_analysis, p.domain, p.translation_direction, p.length_stratum, "
    "m.ko_morpheme_count, m.ko_morpheme_density, m.ko_particle_ratio, "
    "m.ko_ending_ratio, m.ko_deriv_affix_ratio, m.ko_function_morpheme_ratio "
    f"FROM read_parquet('{PILOT_OUTPUT.as_posix()}') m "
    f"JOIN read_parquet('{PAIR_REGISTRY_V002_POSIX}') p USING (pair_id))"
)

def top(order_expr, limit):
    return con.execute(f'SELECT pair_id FROM {pilot_joined} ORDER BY {order_expr} LIMIT {limit}').fetchall()

selected_ids = set()
for domain in ('general', 'dialogue', 'technology', 'other'):
    rows = con.execute(
        f"SELECT pair_id FROM {pilot_joined} WHERE domain = ? LIMIT 8", [domain]
    ).fetchall()
    selected_ids.update(r[0] for r in rows)
for stratum in ('Q1', 'Q5'):
    rows = con.execute(
        f"SELECT pair_id FROM {pilot_joined} WHERE length_stratum = ? LIMIT 6", [stratum]
    ).fetchall()
    selected_ids.update(r[0] for r in rows)
mixed_rows = con.execute(
    f"SELECT p2.pair_id FROM {pilot_joined} p2 "
    f"JOIN read_parquet('{PAIR_REGISTRY_V002_POSIX}') p3 ON p2.pair_id = p3.pair_id "
    "WHERE regexp_matches(p3.ko_text_analysis, '[A-Za-z]') LIMIT 6"
).fetchall()
selected_ids.update(r[0] for r in mixed_rows)
digit_rows = con.execute(
    f"SELECT p2.pair_id FROM {pilot_joined} p2 "
    f"JOIN read_parquet('{PAIR_REGISTRY_V002_POSIX}') p3 ON p2.pair_id = p3.pair_id "
    "WHERE regexp_matches(p3.ko_text_analysis, '[0-9]') LIMIT 6"
).fetchall()
selected_ids.update(r[0] for r in digit_rows)
for order_expr in (
    'ko_morpheme_density DESC', 'ko_morpheme_density ASC',
    'ko_particle_ratio DESC', 'ko_ending_ratio DESC', 'ko_deriv_affix_ratio DESC',
):
    selected_ids.update(r[0] for r in top(order_expr, 6))
con.close()
len(selected_ids)

85

In [9]:
sanity_rows = []
con = duckdb.connect()
meta_rows = con.execute(
    f"SELECT pair_id, ko_text_analysis, domain, translation_direction, length_stratum "
    f"FROM read_parquet('{PAIR_REGISTRY_V002_POSIX}') WHERE pair_id IN (SELECT unnest(?))",
    [list(selected_ids)],
).fetchall()
con.close()
meta_by_id = {row[0]: row for row in meta_rows}

for pair_id in sorted(selected_ids):
    _, ko_text, domain, direction, length_stratum = meta_by_id[pair_id]
    analyzed = kiwi.analyze(ko_text, top_n=1)
    morphs = analyzed[0][0] if analyzed else []
    surface_seq = [m.form for m in morphs]
    pos_seq = [m.tag for m in morphs]
    particle_n = sum(1 for t in pos_seq if t.startswith('J'))
    ending_n = sum(1 for t in pos_seq if t.startswith('E'))
    deriv_n = sum(1 for t in pos_seq if t in ('XSN', 'XSV', 'XSA'))
    morpheme_n = len(morphs)
    eojeol_n = len(ko_text.split())
    warning = 'NONE' if morpheme_n > 0 else 'ZERO_MORPHEME_OUTPUT'
    sanity_rows.append({
        'pair_id': pair_id,
        'domain': domain, 'translation_direction': direction, 'length_stratum': length_stratum,
        'morpheme_surface_sequence': surface_seq,
        'pos_sequence': pos_seq,
        'morpheme_count': morpheme_n,
        'eojeol_count': eojeol_n,
        'particle_count': particle_n,
        'ending_count': ending_n,
        'deriv_affix_count': deriv_n,
        'morpheme_density': morpheme_n / len(ko_text),
        'particle_ratio': particle_n / morpheme_n if morpheme_n else 0.0,
        'ending_ratio': ending_n / morpheme_n if morpheme_n else 0.0,
        'deriv_affix_ratio': deriv_n / morpheme_n if morpheme_n else 0.0,
        'warning': warning,
    })
len(sanity_rows)

85

In [10]:
import json as _json

SANITY_AUDIT_PATH = PROJECT_ROOT / '.runtime/nb04-pilot/MORPHOLOGY_SANITY_AUDIT_SAMPLE.json'
SANITY_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
SANITY_AUDIT_PATH.write_text(_json.dumps(sanity_rows, ensure_ascii=False, indent=2), encoding='utf-8')
{
    'sanity_audit_path': str(SANITY_AUDIT_PATH.relative_to(PROJECT_ROOT)),
    'n_rows': len(sanity_rows),
    'domain_coverage': sorted({r['domain'] for r in sanity_rows}),
    'length_stratum_coverage': sorted({r['length_stratum'] for r in sanity_rows}),
    'zero_morpheme_warnings': sum(1 for r in sanity_rows if r['warning'] != 'NONE'),
}

{'sanity_audit_path': '.runtime/nb04-pilot/MORPHOLOGY_SANITY_AUDIT_SAMPLE.json',
 'n_rows': 85,
 'domain_coverage': ['dialogue', 'general', 'other', 'technology'],
 'length_stratum_coverage': ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
 'zero_morpheme_warnings': 0}

## 6 POS contract verification

sanity audit 표본에서 실제로 관측된 Kiwi POS tag가 primary mapping(`J*`→particle, `E*`→ending, `XSN/XSV/XSA`→deriv affix)과 정확히 일치하는지 확인한다.

In [11]:
observed_tags = sorted({tag for row in sanity_rows for tag in row['pos_sequence']})
particle_tags_observed = sorted(t for t in observed_tags if t.startswith('J'))
ending_tags_observed = sorted(t for t in observed_tags if t.startswith('E'))
deriv_tags_observed = sorted(t for t in observed_tags if t in ('XSN', 'XSV', 'XSA'))
other_tags_observed = sorted(
    t for t in observed_tags
    if not t.startswith('J') and not t.startswith('E') and t not in ('XSN', 'XSV', 'XSA')
)

mismatch = 0
for row in sanity_rows:
    recount_particle = sum(1 for t in row['pos_sequence'] if t.startswith('J'))
    recount_ending = sum(1 for t in row['pos_sequence'] if t.startswith('E'))
    recount_deriv = sum(1 for t in row['pos_sequence'] if t in ('XSN', 'XSV', 'XSA'))
    if (recount_particle, recount_ending, recount_deriv) != (
        row['particle_count'], row['ending_count'], row['deriv_affix_count']
    ):
        mismatch += 1
assert mismatch == 0

{
    'particle_tags_observed': particle_tags_observed,
    'ending_tags_observed': ending_tags_observed,
    'deriv_affix_tags_observed': deriv_tags_observed,
    'other_tags_observed_count': len(other_tags_observed),
    'pos_contract_mismatch_rows': mismatch,
}

{'particle_tags_observed': ['JC',
  'JKB',
  'JKC',
  'JKG',
  'JKO',
  'JKQ',
  'JKS',
  'JKV',
  'JX'],
 'ending_tags_observed': ['EC', 'EF', 'EP', 'ETM', 'ETN'],
 'deriv_affix_tags_observed': ['XSA', 'XSN', 'XSV'],
 'other_tags_observed_count': 27,
 'pos_contract_mismatch_rows': 0}

## 7 Kiwi configuration (record)

```
kiwipiepy        = 0.23.2
Kiwi model       = 0.23.0
user dictionary  = NONE
custom domain dictionary = NONE
input text       = canonical ko_text_analysis (Phase-2 output)
```

Kiwi가 반환한 형태소 segmentation은 언어학적 분석 결과이며, o200k_base tokenizer의 입력으로 재사용하지 않는다. 형태소 분석 ≠ regex chunking ≠ tokenizer BPE token 경계 — 세 layer는 서로 다른 목적의 독립적 측정이다.

## 8 Status

```
PILOT                    : N=1,000, PASS
QUANTITATIVE_DIAGNOSTICS : COMPLETE
SANITY_AUDIT_SAMPLE      : COMPLETE (local-only artifact)
POS_CONTRACT_VERIFICATION: COMPLETE
FULL POPULATION (N=3,836,013): NOT YET EXECUTED
```

Full morphology population run은 이 노트북의 범위 밖이며, G1 close 이후 별도 승인된 clean execution turn에서 진행한다.